### Data Ingestion

In [34]:
### document data structure
from langchain_core.documents import Document

In [21]:
doc=Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source":"exmaple.txt",
        "pages":1,
        "author":"Krish Naik",
        "date_created":"2025-01-01"
    }
)
doc

NameError: name 'Document' is not defined

In [23]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../Data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-13T12:16:20+00:00', 'source': '../Data/pdf/Govt Platform PRD.pdf', 'file_path': '../Data/pdf/Govt Platform PRD.pdf', 'total_pages': 10, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-04-25T22:47:13+05:30', 'trapped': '', 'modDate': "D:20250425224713+05'30'", 'creationDate': 'D:20250413121620Z', 'page': 0}, page_content='Key Features of the Proposed Platform\nNatural Language Q&A:\nUsers can ask questions about government schemes and poli-\ncies as if they are talking to a knowledgeable guide. For example, a user could ask, “What\nare the benefits available for pregnant women in Assam?” or “How can I apply for a youth\nstartup loan?” The AI assistant will understand the query’s intent and fetch the relevant\ndetails (eligibility criteria, application process, deadlines, etc.) from its knowledge base to\nprovide a clear, concise ans

In [8]:
type(pdf_documents[0])

langchain_core.documents.base.Document

### Chunking and Embedding

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List


class ChunkingManager:
    """
    Handles document splitting (chunking)
    """

    def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 200):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = None
        self._load_splitter()

    def _load_splitter(self):
        """
        Initialize text splitter
        """
        try:
            self.text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=self.chunk_size,
                chunk_overlap=self.chunk_overlap,
                length_function=len,
                separators=["\n\n", "\n", " ", ""]
            )
            print("Chunking splitter initialized successfully")
        except Exception as e:
            print("Error initializing chunking splitter")
            print(f"Error: {e}")

    def split_documents(self, documents: List):
        """
        Split documents into chunks
        """
        if self.text_splitter is None:
            raise ValueError("Text splitter not initialized")

        split_docs = self.text_splitter.split_documents(documents)

        print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

        if split_docs:
            print("\nExample chunk:")
            print(f"Content: {split_docs[0].page_content[:200]}...")
            print(f"Metadata: {split_docs[0].metadata}")

        return split_docs

In [25]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [26]:
from sentence_transformers import SentenceTransformer


class ModelLoader:
    """
    Responsible only for loading the embedding model
    """
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.model = None

    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully")
        except Exception as e:
            print("Error loading model")
            print(f"Error: {e}")
            self.model = None

        return self.model


class EmbeddingManager:
    """
    Manages embedding operations
    """
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        loader = ModelLoader(self.model_name)
        self.model = loader._load_model()

    def embed_texts(self, texts):
        """
        Generate embeddings for a list of texts
        """
        if self.model is None:
            raise ValueError("Model not loaded. Cannot generate embeddings.")

        return self.model.encode(texts, show_progress_bar=True)

    def embed_query(self, query: str):
        """
        Generate embedding for a single query
        """
        if self.model is None:
            raise ValueError("Model not loaded. Cannot generate embeddings.")

        return self.model.encode([query])[0]
    
    def generate_embedding(self, text: str):
        """
        Generate embedding for a single text
        
        Returns:
        - embedding_str: string representation of numpy array
        - length: dimension of embedding
        """
        if self.model is None:
            raise ValueError("Model not loaded. Cannot generate embeddings.")
        
        # Generate embedding
        embedding = self.model.encode([text])[0]  # shape: (dim,)
        
        # Convert to numpy array (just to be explicit)
        embedding_np = np.array(embedding)
        
        # Convert to string
        embedding_str = np.array2string(embedding_np, separator=',')
        
        # Get length
        length = len(embedding_np)
        
        return embedding_str, length

In [27]:
manager = EmbeddingManager("all-MiniLM-L6-v2")

embedding_str, length = manager.generate_embedding("RAG is powerful")

print("Embedding:", embedding_str[:100])  # preview
print("Length:", length)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5486.37it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully
Embedding: [-2.02206504e-02, 8.51636007e-02, 3.11996415e-02,-2.25858521e-02,
 -1.05607428e-01, 1.82136036e-02, 
Length: 384


### Vectorizing 

In [35]:
import chromadb
from chromadb.config import Settings
from typing import List, Dict, Any


class VectorStoreManager:
    """
    Handles storing and retrieving embeddings using ChromaDB
    """

    def __init__(self, collection_name: str = "pdf_document", persist_directory: str = "./chroma_db"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self.initialise_store()

    def initialise_store(self):
        """
        Initialize chromadb client and collection
        """
        try:
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name
            )

            print(f"ChromaDB initialized. Collection: {self.collection_name}")

        except Exception as e:
            print("Error initializing ChromaDB")
            print(f"Error: {e}")

    def add_documents(self, documents: List[str], embeddings: List[List[float]], metadatas: List[Dict[str, Any]], ids: List[str]):
        """
        Add documents with embeddings to ChromaDB
        """
        try:
            self.collection.add(
                documents=documents,
                embeddings=embeddings,
                metadatas=metadatas,
                ids=ids
            )
            print(f"Added {len(documents)} documents to collection")

        except Exception as e:
            print("Error adding documents")
            print(f"Error: {e}")

    def query(self, query_embedding: List[float], n_results: int = 5):
        """
        Perform similarity search
        """
        try:
            results = self.collection.query(
                query_embeddings=[query_embedding],
                n_results=n_results
            )
            return results

        except Exception as e:
            print("Error during query")
            print(f"Error: {e}")
            return None

    def persist(self):
        """
        Persistence is automatic in newer ChromaDB versions
        """
        print("Persistence handled automatically by ChromaDB")

store = VectorStoreManager()

ChromaDB initialized. Collection: pdf_document


In [39]:
import uuid


class RAGPipeline:
    """
    Orchestrates full RAG ingestion pipeline
    """

    def __init__(self, chunker, embedder, vector_store):
        self.chunker = chunker
        self.embedder = embedder
        self.vector_store = vector_store

    def ingest(self, documents):
        """
        Full pipeline:
        documents → chunks → embeddings → vector DB
        """

        # Step 1: Chunking
        print("\n# Step 1: Chunking documents...")
        split_docs = self.chunker.split_documents(documents)

        if not split_docs:
            print("No documents to process")
            return

        # Step 2: Prepare data
        print("\n# Step 2: Preparing data...")
        texts = [doc.page_content for doc in split_docs]
        metadatas = [doc.metadata for doc in split_docs]
        ids = [str(uuid.uuid4()) for _ in split_docs]

        # Step 3: Embeddings
        print("\n# Step 3: Generating embeddings...")
        embeddings = self.embedder.embed_texts(texts)

        # Step 4: Store in vector DB
        print("\n# Step 4: Storing in vector DB...")
        self.vector_store.add_documents(
            documents=texts,
            embeddings=embeddings.tolist(),
            metadatas=metadatas,
            ids=ids
        )

        # Step 5: Persist
        print("\n# Step 5: Persisting DB...")
        self.vector_store.persist()

        print("\n RAG ingestion pipeline completed successfully!")

In [40]:
# Step 0: Load documents (you already have this)
pdf_documents = dir_loader.load()

# Step 1: Initialize components
chunker = ChunkingManager()
embedder = EmbeddingManager("all-MiniLM-L6-v2")
store = VectorStoreManager()

# Step 2: Create pipeline
pipeline = RAGPipeline(chunker, embedder, store)

# Step 3: Run ingestion
pipeline.ingest(pdf_documents)

Chunking splitter initialized successfully


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7367.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully
ChromaDB initialized. Collection: pdf_document

# Step 1: Chunking documents...
Split 34 documents into 156 chunks

Example chunk:
Content: Key Features of the Proposed Platform
Natural Language Q&A:
Users can ask questions about government schemes and poli-
cies as if they are talking to a knowledgeable guide. For example, a user could a...
Metadata: {'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-13T12:16:20+00:00', 'source': '../Data/pdf/Govt Platform PRD.pdf', 'file_path': '../Data/pdf/Govt Platform PRD.pdf', 'total_pages': 10, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-04-25T22:47:13+05:30', 'trapped': '', 'modDate': "D:20250425224713+05'30'", 'creationDate': 'D:20250413121620Z', 'page': 0}

# Step 2: Preparing data...

# Step 3: Generating embeddings...


Batches: 100%|██████████| 5/5 [00:16<00:00,  3.34s/it]



# Step 4: Storing in vector DB...
Added 156 documents to collection

# Step 5: Persisting DB...
Persistence handled automatically by ChromaDB

 RAG ingestion pipeline completed successfully!


In [41]:
def query_rag(query, embedder, vector_store, top_k=3):
    """
    Test RAG retrieval pipeline
    """

    print(f"\n Query: {query}")

    # Step 1: Convert query → embedding
    query_embedding = embedder.embed_query(query)

    # Step 2: Search in vector DB
    results = vector_store.query(
        query_embedding=query_embedding,
        n_results=top_k
    )

    # Step 3: Display results
    print("\n  Top Retrieved Chunks:\n")

    for i, doc in enumerate(results['documents'][0]):
        print(f"--- Result {i+1} ---")
        print(doc[:300])  # preview
        print("\n")

    return results

In [42]:
query = "What are the key features of the platform?"

results = query_rag(query, embedder, store)


 Query: What are the key features of the platform?

  Top Retrieved Chunks:

--- Result 1 ---
Key Features of the Proposed Platform
Natural Language Q&A:
Users can ask questions about government schemes and poli-
cies as if they are talking to a knowledgeable guide. For example, a user could ask, “What
are the benefits available for pregnant women in Assam?” or “How can I apply for a youth
s


--- Result 2 ---
Key Features of the Proposed Platform
Natural Language Q&A:
Users can ask questions about government schemes and poli-
cies as if they are talking to a knowledgeable guide. For example, a user could ask, “What
are the benefits available for pregnant women in Assam?” or “How can I apply for a youth
s


--- Result 3 ---
– Design the system to scale horizontally by containerizing components (using Ku-
bernetes) and employing auto-scaling strategies.
– Implement monitoring tools (e.g., Prometheus and Grafana) to track usage, la-
tency, and error rates, thereby ensuring robust perfor